# Compare all ROD models

**Code by Ariyan Azami**

Loads every `*_summary.json` produced by the training notebooks and builds a
side-by-side table + charts of accuracy vs. speed vs. size.

**How to use on Kaggle:** after running the training notebooks, download each
`<variant>_summary.json` from their Output and drop them into a `results/`
folder in this repo (or add them as an input dataset), then point `SEARCH_DIRS`
at that folder. Locally, this just reads `results/`.

In [ ]:
from pathlib import Path
import json, glob

# Where to look for <variant>_summary.json files.
SEARCH_DIRS = ["results", "/kaggle/working", "/kaggle/input"]

rows = []
seen = set()
for d in SEARCH_DIRS:
    for fp in glob.glob(str(Path(d) / "**" / "*_summary.json"), recursive=True):
        data = json.loads(Path(fp).read_text())
        key = data.get("model")
        if key and key not in seen:
            seen.add(key); rows.append(data)

assert rows, "No *_summary.json found — run the training notebooks first."
print(f"Loaded {len(rows)} model summaries:", [r['model'] for r in rows])

In [ ]:
import pandas as pd

df = pd.DataFrame(rows)
order = ["model", "params", "mAP50_95", "mAP50", "precision", "recall",
         "avg_ms_per_img", "fps", "epochs", "imgsz"]
df = df[[c for c in order if c in df.columns]].sort_values("mAP50_95", ascending=False)
df["params (M)"] = (df["params"] / 1e6).round(2)
df

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(20, 6))
d = df.set_index("model")

d["mAP50_95"].plot.bar(ax=axes[0], color="#3b7dd8")
axes[0].set_title("mAP50-95 (higher = better)"); axes[0].set_ylabel("mAP50-95")

d["fps"].plot.bar(ax=axes[1], color="#2ca02c")
axes[1].set_title("Throughput FPS (higher = better)"); axes[1].set_ylabel("FPS")

d["params (M)"].plot.bar(ax=axes[2], color="#d8743b")
axes[2].set_title("Parameters (lower = smaller)"); axes[2].set_ylabel("Million params")

for ax in axes:
    ax.tick_params(axis="x", rotation=30)
plt.tight_layout(); plt.show()

In [ ]:
# Accuracy vs. speed trade-off — the money chart
plt.figure(figsize=(9, 7))
for _, r in df.iterrows():
    plt.scatter(r["avg_ms_per_img"], r["mAP50_95"], s=120)
    plt.annotate(r["model"], (r["avg_ms_per_img"], r["mAP50_95"]),
                 textcoords="offset points", xytext=(8, 4))
plt.xlabel("Latency (ms / image)  — lower is better")
plt.ylabel("mAP50-95  — higher is better")
plt.title("ROD models: accuracy vs. latency")
plt.grid(alpha=0.3); plt.tight_layout(); plt.show()

In [ ]:
# Save a combined CSV for the repo / dataset page
out = Path("results"); out.mkdir(exist_ok=True)
df.to_csv(out / "comparison.csv", index=False)
print("Wrote", out / "comparison.csv")